
 Notebook d'exploration des models en partant du plus simple
 https://docs.microsoft.com/fr-fr/learn/modules/analyze-review-sentiment-with-keras/2-build-and-train-a-neural-network
 afin de définir le meilleur model à travailler et monter sur Azure


In [2]:
# Import the requiered libraries
from collections import defaultdict
import os
from pathlib import Path
import time
import json
import sys
from gensim.models.keyedvectors import Vocab
from gensim.models.word2vec import Word2Vec
from gensim.parsing.preprocessing import remove_stopwords
from keras_preprocessing.text import tokenizer_from_json
from matplotlib.pyplot import ylabel
import missingno as msno
from tensorflow.python.util.nest import _yield_value
from tqdm.notebook import tqdm
import logging  
logging.basicConfig(
                    format= "%(levelname)s - %(asctime)s: %(message)s", 
                    datefmt= '%H:%M:%S', 
                    level= logging.INFO
)
import multiprocessing

import pandas as pd
import numpy as np
import re
import gensim
from gensim.parsing.porter import PorterStemmer
import spacy
nlp = spacy.load("en_core_web_sm")


%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

import keras
from keras.preprocessing.text import Tokenizer
from keras.preprocessing import sequence
from keras.models import Sequential, Model
from keras.layers import Embedding, Input, Dropout, Dense, Flatten, LSTM, Bidirectional 
#from keras.layers.embeddings import Embedding


ImportError: cannot import name 'get_config' from 'tensorflow.python.eager.context' (C:\Data\anaconda3\lib\site-packages\tensorflow\python\eager\context.py)

In [ ]:
# Define constants
FILE_TWEETS = 'sentiment140\\training.1600000.processed.noemoticon.csv'
FILE_DF_TRAIN = 'df_train_5000-Augmented'
FILE_DF_VALIDATE = 'df_validate_1000-Augmented'
FILE_DF_TEST = 'df_test_1000-Augmented'

# Compare 
COMPARE_SOLUTIONS = True

# Dictionary size
TOP_WORDS = 5_000
# Max length of the embedding vectors. 
EMBEDDING_LENGTH = 100

# Create the simple dictionary even if it already exists
CREATE_SIMPLE_DICTIONARY = True

# Smiley - http://www.emoticonr.com/emoticons/
SMILEYS = {
            r"O:)"  : ':angel:',
            r"O:-)" : ':angel:',
            r"X("   : ':angry:',
            r">:-(" : ':angry:',
            r">:("  : ':angry:',
            r"X("   : ':angry:',
            r":)"   : ':smile:', 
            r":-)"  : ':smile:',
            r":(("  : ':crying:', 
            r":("   : ':frown:', 
            r"=(("  : ':broken heart',
            r"=("   : ':frown:', 
            r":-("  : ':frown:',
            r":["   : ':frown:', 
            r";)"   : ':winking:', 
            r";^)"  : ':winking:', 
            r";-)"  : ':winking:',
            r",)"   : ':winking:'  , 
            r",-)"  : ':winking:',
            r";p"   : ':sticking tongue out:', 
            r";-p"  : ':sticking tongue out:',
            r":p"   : '::sticking tongue out:', 
            r":-p"  : '::sticking tongue out::',
            r":|"   : ':straight face:', 
            r":-|"  : ':straight face:',
            r"O.o"  : ':confuse:', 
            r" :/"  : ':confuse:',  # Need space due to http://
            r":-/"  : ':confuse:',
            r":'("  : ':crying:', 
            r":'-(" : ':crying:',
            r"<3"   : ':love:',
            r"</3"  : ':broken heart:',
            r":D"   : ':grin:',
            r":-D"  : ':grin:',
            r":*"   : ':kiss:',
            r":-*"  : ':kiss:',
            r":-X"  : ':kiss:',
            r":x"   : ':love:',
}

WANTED_WORKERS = os.cpu_count() - 1
 
ADDED_STOPWORDS = set(['i', 'to'])
STOP_WORDS = gensim.parsing.preprocessing.STOPWORDS.union(ADDED_STOPWORDS)
# We want the negative so we keep no and not
REMOVED_STOPWORDS = set(['no', 'not'])
STOP_WORDS = STOP_WORDS.difference(REMOVED_STOPWORDS)

WORD2VEC_WINDOW = 3


In [ ]:
# Read the data
df_test = pd.read_pickle(
                            Path('./data/')/f"{FILE_DF_TEST}.pkl"
)
df_train = pd.read_pickle(
                            Path('./data/')/f"{FILE_DF_TRAIN}.pkl"
)
df_validate = pd.read_pickle(
                                Path('./data/')/f"{FILE_DF_VALIDATE}.pkl"
)





In [ ]:
# Work on a model
def fit_and_evaluate(model, test):
    scores = model.evaluate(
                            test[0], test[1],
                            verbose= 0
    )
    print(f"AUC est de {100 * scores[1]:.2f} %")
    return scores



In [ ]:
# Create the NN, fit it and display some résults
def create_simple_NN_fit_and_display(train, validate, test, epochs, batch_size):
    """Fit the model and display the results

    Args:
        train (turple): (X_train, y_train)
        validate (turple): (X_validate, y_validate)
        test (turple): (X_test, y_test)
        epochs (int): number of epoch
        batch_size (int): size of the batch

    Returns:
        score (float): score on the test
    """
    # Create the neural network
    embedding_vector_length = 32
    model = Sequential()
    model.add(
                Embedding(
                            input_dim= max(train[0].flatten()) + 1,
                            output_dim= embedding_vector_length,
                            input_length= EMBEDDING_LENGTH
                )
    )
    model.add(Flatten())
    # 16 is totally arbitrary. Other values can be 
    # used.
    model.add(Dense(16, activation= 'relu'))
    model.add(Dense(16, activation= 'relu'))
    model.add(Dense(1,  activation= 'sigmoid'))
    model.compile(
                    loss= 'binary_crossentropy',
                    optimizer= 'adam',
                    metrics= ['AUC']
    )
    print(model.summary())

    hist = model.fit(
                        train[0], train[1],
                        validation_data= validate,
                        epochs= epochs,
                        batch_size= batch_size
    )
    # Plot the results
    auc = hist.history['auc']
    val = hist.history['val_auc']
    epochs = range(1, len(auc) + 1)

    plt.plot(epochs, auc, '-', label= 'Training Area Under the Curve')
    plt.plot(epochs, val, ':', label= 'Validation AUC')
    plt.title('Training and Validation AUC')
    plt.xlabel('Epoch')
    plt.ylabel('AUC')
    plt.legend(loc='upper left')
    plt.plot()
    plt.show()

    scores = model.evaluate(
                            test[0], test[1],
                            verbose= 0
    )
    print(f"AUC est de {100 * scores[1]:.2f} %")
    return scores




In [ ]:
# Create a LSTM, fit it and display some résults
def create_LSTM_fit_and_display(
                                train, validate, 
                                test, 
                                embed_dim, lstm_out, batch_size,
                                epochs, verbose
):
    """Create a LSTM, fit the model and display the results

    Args:
        train (turple): (X_train, y_train)
        validate (turple): (X_validate, y_validate)
        test (turple): (X_test, y_test)
        embed_dim (int): number of embed_dim
        lstm_out (int): number for lstm_out
        batch_size (int): size of the batch
        epochs (int): number of epochs
        verbose (int): verbose value

    Returns:
        score (float): score on the test
    """
    # Input for variable-length sequences of integers
    inputs = keras.Input(shape=(None,), dtype="int32")
    # Embed each integer in a 128-dimensional vector
    x = Embedding(max(train[0].flatten()) + 1, embed_dim)(inputs)
    # Add 2 bidirectional LSTMs
    x = Bidirectional(LSTM(64, return_sequences=True))(x)
    x = Bidirectional(LSTM(64))(x)
    # Add a classifier
    outputs = Dense(1, activation="sigmoid")(x)
    model = keras.Model(inputs, outputs)
    model.compile(
                    loss= 'binary_crossentropy',
                    optimizer= 'adam',
                    metrics= ['AUC']
    )
    model.summary()

    hist = model.fit(
                        train[0], train[1],
                        validation_data= (validate[0], validate[1]),
                        epochs= epochs,
                        batch_size= batch_size,
                        verbose= verbose
    )
    # Plot the results
    auc = hist.history['auc']
    val = hist.history['val_auc']
    epochs = range(1, len(auc) + 1)

    plt.plot(epochs, auc, '-', label= 'Training Area Under the Curve')
    plt.plot(epochs, val, ':', label= 'Validation AUC')
    plt.title('Training and Validation AUC')
    plt.xlabel('Epoch')
    plt.ylabel('AUC')
    plt.legend(loc='upper left')
    plt.plot()
    plt.show()

    scores = model.evaluate(
                            test[0], test[1],
                            verbose= 0
    )
    print(f"AUC est de {100 * scores[1]:.2f} %")
    return scores



In [ ]:
# Create the neural network, fit it and display the results
def create_Glove_fit_and_display(
                                    train,
                                    validate,
                                    test,
                                    epochs,
                                    batch_size,
                                    input_shape,
                                    embedding_layer,
                                    verbose= 0
):
    layer_indices = Input(input_shape)
    embeddings = embedding_layer(layer_indices)
    x = LSTM(128, return_sequences=True)(embeddings)
    x = Dropout(0.6)(x)
    x = LSTM(128, return_sequences=True)(x)
    x = Dropout(0.6)(x)
    x = LSTM(128)(x)
    x = Dense(1, activation='sigmoid')(x)
    model = Model(inputs= layer_indices, outputs= x)

    adam = keras.optimizers.Adam(learning_rate= 0.0001)
    model.compile(
                    optimizer= adam, 
                    loss= 'binary_crossentropy', 
                    metrics= ['AUC']
    )
    model.summary()

    hist = model.fit(
                        train[0], train[1],
                        validation_data= (validate[0], validate[1]),
                        epochs= epochs,
                        batch_size= batch_size,
                        verbose= verbose
    )
    # Plot the results
    auc = hist.history['auc']
    val = hist.history['val_auc']
    epochs = range(1, len(auc) + 1)

    plt.plot(epochs, auc, '-', label= 'Training Area Under the Curve')
    plt.plot(epochs, val, ':', label= 'Validation AUC')
    plt.title('Training and Validation AUC')
    plt.xlabel('Epoch')
    plt.ylabel('AUC')
    plt.legend(loc='upper left')
    plt.plot()
    plt.show()

    scores = model.evaluate(
                            test[0], test[1],
                            verbose= 0
    )
    print(f"AUC est de {100 * scores[1]:.2f} %")
    return scores



In [ ]:
# Read the glove vector
glove_file = os.getcwd() + '\\data\\glove.twitter.27B.50d.txt' 
with open( glove_file, 'r', encoding= 'UTF-8') as file_handler:
    words = set()
    word_to_vec_glove_vector = {}
    for line in glove_file:
        w_line = line.split()
        curr_word = w_line[0]
        word_to_vec_glove_vector[curr_word] = np.array(w_line[1:], dtype=np.float64)
glove_embed_vector_len = word_to_vec_glove_vector['moon'].shape[0]
logging.info("Glove chargé et le vecteur est réalisé.")



 Nous testons différentes solutions pour voir les résultats


In [ ]:
# Loop on the different models
data_to_be_tested = [
                        'w2f_clean', 
                        'w2f_stem', 
                        'w2f_lemma',

]

progress_bar = tqdm(
                    total= len(data_to_be_tested),
                    desc= 'Tests models',
                    leave= False
)
memo_score = {
                'data_kind'      : [],
                'model'          : [],
                'paramters'      : [],
                'score'          : [],
                'execution_time' : []
}
for kind_data in data_to_be_tested:
    progress_bar.update()
    print(f"Nous travaillons sur {kind_data}")
    X_train = df_train[kind_data]
    y_train = df_train['neg']
    X_validate = df_validate[kind_data]
    y_validate = df_validate['neg']
    X_test = df_test[kind_data]
    y_test = df_test['neg']

    logging.info(f"Simple {kind_data}")
    memo_score['data_kind'].append(kind_data)
    memo_score['model'].append('Simple')
    memo_score['parameters'].append({'epochs' : 15, 'batch_size' : 128})
    memo_score['score'].append(score)
    time_in = time.time()
    score = create_simple_NN_fit_and_display(
                        train= (X_train, y_train),
                        validate= (X_validate, y_validate),
                        test= (X_test, y_test),
                        epochs= memo_score['parameters'][-1]['epochs'],
                        batch_size= memo_score['parameters'][-1]['batch_size']
    )
    memo_score['execution_time'].append(time.time() - time_in)

    memo_score['data_kind'].append(kind_data)
    memo_score['model'].append('Simple')
    memo_score['parameters'].append({'epochs' : 4, 'batch_size' : 128})
    memo_score['score'].append(score)
    time_in = time.time()
    score = create_simple_NN_fit_and_display(
                        train= (X_train, y_train),
                        validate= (X_validate, y_validate),
                        test= (X_test, y_test),
                        epochs= memo_score['parameters'][-1]['epochs'],
                        batch_size= memo_score['parameters'][-1]['batch_size']
    )
    memo_score['execution_time'].append(time.time() - time_in)

    memo_score['data_kind'].append(kind_data)
    memo_score['model'].append('Simple')
    memo_score['parameters'].append({'epochs' : 5, 'batch_size' : 32})
    memo_score['score'].append(score)
    time_in =time.time()
    score = create_simple_NN_fit_and_display(
                        train= (X_train, y_train),
                        validate= (X_validate, y_validate),
                        test= (X_test, y_test),
                        epochs= memo_score['parameters'][-1]['epochs'],
                        batch_size= memo_score['parameters'][-1]['batch_size']
    )
    memo_score['execution_time'].append(time.time() - time_in)

    # Compute LSTM
    logging.info(f"LSTM {kind_data}")
    memo_score['data_kind'].append(kind_data)
    memo_score['model'].append('LSTM')
    memo_score['parameters'].append({'epochs' : 10, 'batch_size' : 128})
    memo_score['score'].append(score)
    time_in = time.time()
    score = create_LSTM_fit_and_display(
                        train= (X_train, y_train),
                        validate= (X_validate, y_validate),
                        test= (X_test, y_test),
                        embed_dim= 128,
                        lstm_out= 200, 
                        epochs= memo_score['parameters'][-1]['epochs'],
                        batch_size= memo_score['parameters'][-1]['batch_size'],
                        verbose= 4,
    )
    memo_score['execution_time'].append(time.time() - time_in)

    memo_score['data_kind'].append(kind_data)
    memo_score['model'].append('LSTM')
    memo_score['parameters'].append({'epochs' : 7, 'batch_size' : 32})
    memo_score['score'].append(score)
    time_in = time.time()
    score = create_LSTM_fit_and_display(
                        train= (X_train, y_train),
                        validate= (X_validate, y_validate),
                        test= (X_test, y_test),
                        embed_dim= 128,
                        lstm_out= 200, 
                        epochs= memo_score['parameters'][-1]['epochs'],
                        batch_size= memo_score['parameters'][-1]['batch_size'],
                        verbose= 4,
    )
    memo_score['execution_time'].append(time.time() - time_in)

    # Compute the Glove
    logging.info(f"Glove {kind_data}")
    vocab_len = len(
                    set(
                        [
                            df_test[kind_data], 
                            df_validate[kind_data],
                            df_train[kind_data]
                        ].flatten()
                    )
    )
    emb_matrix = np.zeros(vocab_len, glove_embed_vector_len)
    progress_bar = tqdm(
                        total= len(word_to_vec_glove_vector.item()),
                        desc= 'glove',
                        leave= False
    )
    for word, index in word_to_vec_glove_vector.items():
        progress_bar.update()
        embedding_vector = word_to_vec_glove_vector.get(word)
        if embedding_vector is not None:
            emb_matrix[index, :] = embedding_vector
    progress_bar.close()

    embedding_layer = Embedding(
                                input_dim= vocab_len,
                                output_dim= glove_embed_vector_len,
                                input_length= EMBEDDING_LENGTH,
                                weights= [emb_matrix],
                                trainable= False
    )

    memo_score['data_kind'].append(kind_data)
    memo_score['model'].append('Glove')
    memo_score['parameters'].append({'epochs' : 15, 'batch_size' : 128})
    memo_score['score'].append(score)
    time_in = time.time()
    score = create_Glove_fit_and_display(
                        train= (X_train, y_train),
                        validate= (X_validate, y_validate),
                        test= (X_test, y_test),
                        epochs= memo_score['parameters'][-1]['epochs'],
                        batch_size= memo_score['parameters'][-1]['batch_size'],
                        input_shape= (None,),
                        embedding_layer= embedding_layer,
    )
    memo_score['execution_time'].append(time.time() - time_in)

    memo_score['data_kind'].append(kind_data)
    memo_score['model'].append('Glove')
    memo_score['parameters'].append({'epochs' : 15, 'batch_size' : 32})
    memo_score['score'].append(score)
    time_in = time.time()
    score = create_Glove_fit_and_display(
                        train= (X_train, y_train),
                        validate= (X_validate, y_validate),
                        test= (X_test, y_test),
                        epochs= memo_score['parameters'][-1]['epochs'],
                        batch_size= memo_score['parameters'][-1]['batch_size'],
                        input_shape= (None,),
                        embedding_layer= embedding_layer,
    )
    memo_score['execution_time'].append(time.time() - time_in)





 Nous avons maintenant une comparaison des différentes solutions.
 Nous choisissons donc de fabriquer un model de type Glove

In [ ]:
# Other implementation of Glove
from gensim.scripts.glove2word2vec import glove2word2vec
from gensim.models import KeyedVectors

glove2word2vec(
                Path('./data/')/'glove.twitter.27B.50d.txt',
                word2vec_output_file= 
                        Path('./data/')/'glove.twitter.27B.50d.txt.word2vec'
)

w2v_model = glove2word2vec.load_word2vec_format(
                        Path('./data/')/'glove.twitter.27B.50d.txt.word2vec',
                        binary= False
)
score = w2v_model.evaluate(
                            df_test['clean'], df_test['neg']
)
print(f"AUC est de {100 * score[1]:.2f} %")
score = w2v_model.evaluate(
                            df_test['stem'], df_test['neg']
)
print(f"AUC est de {100 * score[1]:.2f} %")
score = w2v_model.evaluate(
                            df_test['lemma'], df_test['neg']
)
print(f"AUC est de {100 * score[1]:.2f} %")

# REgarder Streamlit
